In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [ ]:
!pip install -q monai nibabel tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

root_path = "/content/drive/MyDrive/brain segmentation/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

print("Total Patients:", len(os.listdir(root_path)))

Total Patients: 1241


In [ ]:
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import UNet
from tqdm import tqdm
import random

In [ ]:
patients = sorted(os.listdir(root_path))[:500]

random.seed(42)
random.shuffle(patients)

train_patients = patients[:int(0.8 * len(patients))]
val_patients = patients[int(0.8 * len(patients)):]

print("Train:", len(train_patients))
print("Val:", len(val_patients))

Train: 400
Val: 100


In [ ]:
class BraTSDataset(Dataset):
    def __init__(self, root_path, patient_list):
        self.root_path = root_path
        self.patient_list = patient_list

    def __len__(self):
        return len(self.patient_list)

    def normalize(self, x):
        return (x - np.mean(x)) / (np.std(x) + 1e-5)

    def center_crop(self, x, size=128):
        z, y, x_dim = x.shape
        z1 = (z - size)//2
        y1 = (y - size)//2
        x1 = (x_dim - size)//2
        return x[z1:z1+size, y1:y1+size, x1:x1+size]

    def __getitem__(self, idx):
      patient = self.patient_list[idx]
      patient_path = os.path.join(self.root_path, patient)

      try:
        t1  = nib.load(os.path.join(patient_path, f"{patient}-t1n.nii.gz")).get_fdata()
        t1c = nib.load(os.path.join(patient_path, f"{patient}-t1c.nii.gz")).get_fdata()
        t2f = nib.load(os.path.join(patient_path, f"{patient}-t2f.nii.gz")).get_fdata()
        t2w = nib.load(os.path.join(patient_path, f"{patient}-t2w.nii.gz")).get_fdata()
        seg = nib.load(os.path.join(patient_path, f"{patient}-seg.nii.gz")).get_fdata()
      except:
        print(f"❌ Skipping corrupted file: {patient}")
      return self.__getitem__((idx + 1) % len(self.patient_list))
      t1  = self.center_crop(self.normalize(t1))
      t1c = self.center_crop(self.normalize(t1c))
      t2f = self.center_crop(self.normalize(t2f))
      t2w = self.center_crop(self.normalize(t2w))
      seg = self.center_crop(seg)

      volume = np.stack([t1, t1c, t2f, t2w], axis=0)
      return (
          torch.tensor(volume, dtype=torch.float32),
          torch.tensor(seg, dtype=torch.long)
    )

In [ ]:
train_dataset = BraTSDataset(root_path, train_patients)
val_dataset = BraTSDataset(root_path, val_patients)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)

print("Train Batches:", len(train_loader))

Train Batches: 400


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
).to(device)

In [ ]:
from monai.losses import DiceCELoss

loss_fn = DiceCELoss(
    to_onehot_y=True,
    softmax=True
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def dice_score(pred, target, num_classes=4, epsilon=1e-6):
    pred = torch.argmax(pred, dim=1)

    dice = 0
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice += (2. * intersection + epsilon) / (union + epsilon)

    return dice / (num_classes - 1)

In [ ]:
# ================== MEMORY SAFE EVALUATION ==================

import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- LOAD MODEL --------
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
).to(device)

model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pth"))
model.eval()

print("✅ Model loaded")

# -------- METRIC ACCUMULATORS --------
all_accuracy = []
all_precision = []
all_recall = []
all_f1 = []

# -------- CONFUSION MATRIX --------
num_classes = 4
conf_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

# -------- LOOP --------
with torch.no_grad():
    for volume, mask in val_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        outputs = model(volume)
        preds = torch.argmax(outputs, dim=1)

        preds_np = preds.cpu().numpy().flatten()
        mask_np = mask.cpu().numpy().flatten()

        # Compute batch metrics
        all_accuracy.append(accuracy_score(mask_np, preds_np))
        all_precision.append(precision_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_recall.append(recall_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_f1.append(f1_score(mask_np, preds_np, average='weighted', zero_division=0))

        # Update confusion matrix
        for t, p in zip(mask_np, preds_np):
            conf_matrix[t, p] += 1

# -------- FINAL METRICS --------
accuracy = np.mean(all_accuracy)
precision = np.mean(all_precision)
recall = np.mean(all_recall)
f1 = np.mean(all_f1)

print("\n📊 FINAL RESULTS:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# -------- CONFUSION MATRIX PLOT --------
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.imshow(conf_matrix)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -------- GRAPH --------
metrics = ["Accuracy", "Precision", "Recall", "F1"]
values = [accuracy, precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.show()

✅ Model loaded


In [ ]:
# ================== MEMORY SAFE EVALUATION ==================

import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- LOAD MODEL --------
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
).to(device)

model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pth"))
model.eval()

print("✅ Model loaded")

# -------- METRIC ACCUMULATORS --------
all_accuracy = []
all_precision = []
all_recall = []
all_f1 = []

# -------- CONFUSION MATRIX --------
num_classes = 4
conf_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

# -------- LOOP --------
with torch.no_grad():
    for volume, mask in val_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        outputs = model(volume)
        preds = torch.argmax(outputs, dim=1)

        preds_np = preds.cpu().numpy().flatten()
        mask_np = mask.cpu().numpy().flatten()

        # Compute batch metrics
        all_accuracy.append(accuracy_score(mask_np, preds_np))
        all_precision.append(precision_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_recall.append(recall_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_f1.append(f1_score(mask_np, preds_np, average='weighted', zero_division=0))

        # Update confusion matrix
        for t, p in zip(mask_np, preds_np):
            conf_matrix[t, p] += 1

# -------- FINAL METRICS --------
accuracy = np.mean(all_accuracy)
precision = np.mean(all_precision)
recall = np.mean(all_recall)
f1 = np.mean(all_f1)

print("\n📊 FINAL RESULTS:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# -------- CONFUSION MATRIX PLOT --------
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.imshow(conf_matrix)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -------- GRAPH --------
metrics = ["Accuracy", "Precision", "Recall", "F1"]
values = [accuracy, precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.show()

In [ ]:
# ================== MEMORY SAFE EVALUATION ==================

import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- LOAD MODEL --------
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
).to(device)

model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pth"))
model.eval()

print("✅ Model loaded")

# -------- METRIC ACCUMULATORS --------
all_accuracy = []
all_precision = []
all_recall = []
all_f1 = []

# -------- CONFUSION MATRIX --------
num_classes = 4
conf_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

# -------- LOOP --------
with torch.no_grad():
    for volume, mask in val_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        outputs = model(volume)
        preds = torch.argmax(outputs, dim=1)

        preds_np = preds.cpu().numpy().flatten()
        mask_np = mask.cpu().numpy().flatten()

        # Compute batch metrics
        all_accuracy.append(accuracy_score(mask_np, preds_np))
        all_precision.append(precision_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_recall.append(recall_score(mask_np, preds_np, average='weighted', zero_division=0))
        all_f1.append(f1_score(mask_np, preds_np, average='weighted', zero_division=0))

        # Update confusion matrix
        for t, p in zip(mask_np, preds_np):
            conf_matrix[t, p] += 1

# -------- FINAL METRICS --------
accuracy = np.mean(all_accuracy)
precision = np.mean(all_precision)
recall = np.mean(all_recall)
f1 = np.mean(all_f1)

print("\n📊 FINAL RESULTS:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# -------- CONFUSION MATRIX PLOT --------
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.imshow(conf_matrix)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -------- GRAPH --------
metrics = ["Accuracy", "Precision", "Recall", "F1"]
values = [accuracy, precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.show()# ================== RESUME TRAINING ==================

import torch
from monai.networks.nets import UNet
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- MODEL --------
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
).to(device)

# -------- OPTIMIZER --------
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# -------- LOAD CHECKPOINT --------
CKPT_PATH = "/content/drive/MyDrive/checkpoint.pth"

checkpoint = torch.load(CKPT_PATH)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1
best_dice = checkpoint['best_dice']

print("✅ Resuming from epoch:", start_epoch)
print("🔥 Best Dice so far:", best_dice)

# -------- SCALER --------
scaler = GradScaler()

# -------- TOTAL EPOCHS --------
EPOCHS = 50

# ================== TRAINING ==================
for epoch in range(start_epoch, EPOCHS):

    # -------- TRAIN --------
    model.train()
    total_loss = 0

    for volume, mask in train_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(volume)
            loss = loss_fn(outputs, mask.unsqueeze(1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()
    dice_total = 0

    with torch.no_grad():
        for volume, mask in val_loader:
            volume = volume.to(device)
            mask = mask.to(device)

            outputs = model(volume)
            dice_total += dice_score(outputs, mask).item()

    avg_val_dice = dice_total / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Dice: {avg_val_dice:.4f}")
    print("-" * 30)

    # -------- SAVE BEST MODEL --------
    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_model.pth")
        print(f"✅ Best model saved! Dice: {best_dice:.4f}")

    # -------- SAVE CHECKPOINT --------
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_dice': best_dice,
    }, CKPT_PATH)

print("🎉 Training resumed successfully!")
print("Final Best Dice:", best_dice)

✅ Resuming from epoch: 24
🔥 Best Dice so far: 0.7761751905083656


/tmp/ipykernel_976/2641104463.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_976/2641104463.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 25/50
Train Loss: 0.2088
Val Dice: 0.7730
------------------------------
Epoch 26/50
Train Loss: 0.1808
Val Dice: 0.7874
------------------------------
✅ Best model saved! Dice: 0.7874
Epoch 27/50
Train Loss: 0.1720
Val Dice: 0.7861
------------------------------
Epoch 28/50
Train Loss: 0.1611
Val Dice: 0.7945
------------------------------
✅ Best model saved! Dice: 0.7945
Epoch 29/50
Train Loss: 0.1552
Val Dice: 0.7833
------------------------------
Epoch 30/50
Train Loss: 0.1556
Val Dice: 0.7935
------------------------------
Epoch 31/50
Train Loss: 0.1557
Val Dice: 0.7894
------------------------------
Epoch 32/50
Train Loss: 0.1486
Val Dice: 0.7948
------------------------------
✅ Best model saved! Dice: 0.7948
Epoch 33/50
Train Loss: 0.1424
Val Dice: 0.8049
------------------------------
✅ Best model saved! Dice: 0.8049
Epoch 34/50
Train Loss: 0.1437
Val Dice: 0.7929
------------------------------
Epoch 35/50
Train Loss: 0.1363
Val Dice: 0.8041
------------------------------

In [ ]:
EPOCHS = 30

train_losses = []
val_dice_scores = []
best_dice = 0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    model.train()
    total_loss = 0

    for volume, mask in train_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        outputs = model(volume)

        loss = loss_fn(outputs, mask.unsqueeze(1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)


    # ------------------ VALIDATION ------------------
    # ------------------ VALIDATION ------------------
    model.eval()
    dice_total = 0

    with torch.no_grad():
        for volume, mask in val_loader:
            volume = volume.to(device)
            mask = mask.to(device)

            outputs = model(volume)
            dice_total += dice_score(outputs, mask).item()

    avg_val_dice = dice_total / len(val_loader)
    val_dice_scores.append(avg_val_dice)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Dice: {avg_val_dice:.4f}")
    print("-" * 30)

    # Save best model
    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save(model.state_dict(), "best_model.pth")

In [ ]:
model.eval()
dice_total = 0

with torch.no_grad():
    for volume, mask in tqdm(val_loader):
        volume = volume.to(device)
        mask = mask.to(device)
        outputs = model(volume)
        dice_total += dice_score(outputs, mask).item()

print("Validation Dice:", dice_total / len(val_loader))

In [ ]:
def dice_score(pred, target, num_classes=4, epsilon=1e-6):
    pred = torch.argmax(pred, dim=1)

    dice = 0
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice += (2. * intersection + epsilon) / (union + epsilon)

    return dice / (num_classes - 1)

In [ ]:
model.eval()
dice_total = 0

with torch.no_grad():
    for volume, mask in tqdm(val_loader):
        volume = volume.to(device)
        mask = mask.to(device)
        outputs = model(volume)
        dice_total += dice_score(outputs, mask).item()

print("Validation Dice:", dice_total / len(val_loader))

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/brats_3d_unet_full.pth")

In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/Colab Notebooks/brats_3d_unet_full.pth"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

# Get one validation sample
volume, mask = val_dataset[0]

volume = volume.unsqueeze(0).to(device)

with torch.no_grad():
    output = model(volume)
    pred = torch.argmax(output, dim=1).cpu().numpy()[0]

# Move MRI back to CPU
volume_np = volume.cpu().numpy()[0]

# Select middle slice
slice_idx = volume_np.shape[-1] // 2

flair_slice = volume_np[3, :, :, slice_idx]  # FLAIR channel
pred_slice = pred[:, :, slice_idx]

plt.figure(figsize=(8,8))
plt.title("Tumor Segmentation Overlay")

# Show MRI
plt.imshow(flair_slice, cmap="gray")

# Overlay prediction (only tumor regions)
overlay = np.zeros((*pred_slice.shape, 4))  # RGBA

# Class colors
# 1 = edema (yellow)
# 2 = tumor core (red)
# 3 = enhancing tumor (cyan)

overlay[pred_slice == 1] = [1, 1, 0, 0.4]
overlay[pred_slice == 2] = [1, 0, 0, 0.5]
overlay[pred_slice == 3] = [0, 1, 1, 0.5]

plt.imshow(overlay)
plt.axis("off")
plt.show()

In [ ]:
plt.plot(train_losses)
plt.title("Training Loss")
plt.savefig("train_loss.png")

plt.plot(val_dice_scores)
plt.title("Validation Dice")
plt.savefig("val_dice.png")

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
).to(device)

model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/brats_3d_unet_full.pth"))
model.eval()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

all_preds = []
all_targets = []

with torch.no_grad():
    for volume, mask in val_loader:
        volume = volume.to(device)
        mask = mask.to(device)

        outputs = model(volume)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy().flatten())
        all_targets.extend(mask.cpu().numpy().flatten())

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
print(cm)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

accuracy = accuracy_score(all_targets, all_preds)
precision = precision_score(all_targets, all_preds, average='weighted')
recall = recall_score(all_targets, all_preds, average='weighted')
f1 = f1_score(all_targets, all_preds, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]
values = [accuracy, precision, recall, f1]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Performance")
plt.show()